# Module 19: Week 8 BBO Optimization

## LLM-Centred Strategy Analysis

With 17 data points, we now have sufficient data to:
1. Build more reliable GP models with proper length scale estimation
2. Use multi-kernel ensembles to handle unknown smoothness
3. Apply TuRBO-style trust regions for exploitation
4. Consider prompt/context effects on optimization decisions

### Week 7 Results Analysis
- **F3**: NEW BEST! -0.0145 (improved from -0.035)
- **F6**: NEW BEST! -0.681 (improved from -0.714)
- F1, F2, F4, F5, F7, F8: Regression or minimal change

### Strategy for Week 8
- Exploit best regions for F3, F6, F7, F8 (showing improvement potential)
- Try alternative regions for stagnant functions (F2)
- Use tight trust regions for near-optimal functions (F1, F5)
- Apply ensemble methods with different kernels

In [6]:
import numpy as np
import pandas as pd
from sklearn.gaussian_process import GaussianProcessRegressor
from sklearn.gaussian_process.kernels import Matern, ConstantKernel, RBF, WhiteKernel
from sklearn.preprocessing import StandardScaler
from scipy.optimize import minimize
from scipy.stats import norm
import warnings
warnings.filterwarnings('ignore')

np.random.seed(42)

import sys
sys.path.append('..')
from src.utils import load_data, save_submission

In [7]:
# Helper function to avoid numpy round issues
def fmt(arr, d=4):
    return [round(float(x), d) for x in arr]

# Load all data and summarize
best_values = {}
best_locations = {}
all_data = {}

for func_id in range(1, 9):
    df = load_data(func_id)
    all_data[func_id] = df
    
    # Get dimension count
    dim = len([c for c in df.columns if c.startswith('x')])
    
    # Find best
    best_idx = df['y'].idxmax()
    best_y = df.loc[best_idx, 'y']
    best_x = df.loc[best_idx, [f'x{i}' for i in range(dim)]].values
    
    best_values[func_id] = best_y
    best_locations[func_id] = best_x
    
    print(f"F{func_id} ({dim}D): Best = {best_y:.4f} at {fmt(best_x)}")
    print(f"   Total points: {len(df)}, Source: {df.loc[best_idx, 'source']}")

F1 (2D): Best = 1.6263 at [0.6346, 0.6356]
   Total points: 17, Source: week_5_submission
F2 (2D): Best = 0.6670 at [0.7026, 0.9266]
   Total points: 17, Source: week_4_submission
F3 (3D): Best = -0.0145 at [0.5198, 0.6294, 0.3797]
   Total points: 22, Source: week_7_submission
F4 (4D): Best = 0.5995 at [0.4046, 0.4148, 0.3574, 0.399]
   Total points: 37, Source: week_1_submission
F5 (4D): Best = 1618.4931 at [0.3627, 0.2734, 0.9961, 0.9975]
   Total points: 27, Source: week_1_submission
F6 (5D): Best = -0.6808 at [0.7084, 0.1454, 0.7533, 0.7305, 0.0527]
   Total points: 27, Source: week_7_submission
F7 (6D): Best = 2.4033 at [0.01, 0.1564, 0.5383, 0.2527, 0.3992, 0.7464]
   Total points: 37, Source: week_5_submission
F8 (8D): Best = 9.9151 at [0.0251, 0.095, 0.163, 0.0358, 0.8874, 0.3193, 0.1665, 0.2045]
   Total points: 47, Source: week_6_submission


## Advanced Multi-Kernel GP Ensemble

We use an ensemble of GPs with different Matern smoothness parameters (ν = 0.5, 1.5, 2.5) weighted by model evidence.

In [8]:
class MultiKernelGPEnsemble:
    """Ensemble of GPs with different Matern smoothness parameters."""
    
    def __init__(self, dim):
        self.dim = dim
        self.gps = []
        self.weights = []
        self.scaler_X = StandardScaler()
        self.scaler_y = StandardScaler()
        
    def fit(self, X, y):
        # Normalize inputs and outputs
        X_scaled = self.scaler_X.fit_transform(X)
        y_scaled = self.scaler_y.fit_transform(y.reshape(-1, 1)).ravel()
        
        # Different smoothness levels
        nus = [0.5, 1.5, 2.5]
        self.gps = []
        log_marginal_likelihoods = []
        
        for nu in nus:
            kernel = ConstantKernel(1.0, (1e-3, 1e3)) * Matern(
                length_scale=np.ones(self.dim), 
                length_scale_bounds=(1e-2, 1e2),
                nu=nu
            ) + WhiteKernel(noise_level=1e-3, noise_level_bounds=(1e-10, 1e1))
            
            gp = GaussianProcessRegressor(
                kernel=kernel, 
                n_restarts_optimizer=10,
                normalize_y=False,
                alpha=1e-6
            )
            gp.fit(X_scaled, y_scaled)
            self.gps.append(gp)
            log_marginal_likelihoods.append(gp.log_marginal_likelihood_value_)
        
        # Compute weights from log marginal likelihoods
        lml = np.array(log_marginal_likelihoods)
        lml = lml - np.max(lml)  # Numerical stability
        weights = np.exp(lml)
        self.weights = weights / np.sum(weights)
        
        print(f"  Kernel weights: ν=0.5: {self.weights[0]:.3f}, ν=1.5: {self.weights[1]:.3f}, ν=2.5: {self.weights[2]:.3f}")
        
    def predict(self, X, return_std=True):
        X_scaled = self.scaler_X.transform(X)
        
        mu_ensemble = np.zeros(len(X))
        var_ensemble = np.zeros(len(X))
        
        for gp, w in zip(self.gps, self.weights):
            mu, sigma = gp.predict(X_scaled, return_std=True)
            mu_ensemble += w * mu
            var_ensemble += w * (sigma**2 + mu**2)
        
        var_ensemble -= mu_ensemble**2
        sigma_ensemble = np.sqrt(np.maximum(var_ensemble, 1e-10))
        
        # Transform back
        mu_orig = self.scaler_y.inverse_transform(mu_ensemble.reshape(-1, 1)).ravel()
        sigma_orig = sigma_ensemble * self.scaler_y.scale_[0]
        
        if return_std:
            return mu_orig, sigma_orig
        return mu_orig
    
    def sample_posterior(self, X, n_samples=100):
        """Thompson sampling via posterior samples."""
        X_scaled = self.scaler_X.transform(X)
        
        # Sample from each GP proportionally to weights
        all_samples = []
        for gp, w in zip(self.gps, self.weights):
            n_from_this = max(1, int(n_samples * w))
            samples = gp.sample_y(X_scaled, n_samples=n_from_this)
            all_samples.append(samples)
        
        combined = np.hstack(all_samples)
        # Transform back
        combined = combined * self.scaler_y.scale_[0] + self.scaler_y.mean_[0]
        return combined

## Acquisition Functions

We use multiple acquisition functions and combine their recommendations:
1. **UCB** with adaptive kappa
2. **Expected Improvement (EI)** 
3. **Thompson Sampling**
4. **Probability of Improvement (PI)** with small xi for exploitation

In [9]:
def ucb(mu, sigma, kappa=2.0):
    """Upper Confidence Bound."""
    return mu + kappa * sigma

def expected_improvement(mu, sigma, y_best, xi=0.01):
    """Expected Improvement."""
    improvement = mu - y_best - xi
    Z = improvement / (sigma + 1e-10)
    ei = improvement * norm.cdf(Z) + sigma * norm.pdf(Z)
    return np.where(sigma > 1e-10, ei, 0.0)

def probability_of_improvement(mu, sigma, y_best, xi=0.001):
    """Probability of Improvement - good for exploitation."""
    Z = (mu - y_best - xi) / (sigma + 1e-10)
    return norm.cdf(Z)

def thompson_sampling(gp_ensemble, X_candidates, n_samples=50):
    """Thompson Sampling via posterior sample optimization."""
    samples = gp_ensemble.sample_posterior(X_candidates, n_samples=n_samples)
    
    # For each sample, find the best point
    best_indices = []
    for i in range(samples.shape[1]):
        best_idx = np.argmax(samples[:, i])
        best_indices.append(best_idx)
    
    # Return most frequently selected point
    from collections import Counter
    counter = Counter(best_indices)
    return counter.most_common(1)[0][0]

## TuRBO-Style Trust Region Optimization

Adaptive trust regions based on recent success/failure rates.

In [10]:
class TuRBOTrustRegion:
    """Trust region management inspired by TuRBO algorithm."""
    
    def __init__(self, dim, center, initial_radius=0.05, min_radius=0.01, max_radius=0.3):
        self.dim = dim
        self.center = np.array(center)
        self.radius = initial_radius
        self.min_radius = min_radius
        self.max_radius = max_radius
        self.success_count = 0
        self.failure_count = 0
        
    def generate_candidates(self, n_candidates=10000):
        """Generate candidates within trust region using Sobol-like quasi-random."""
        # Use Latin Hypercube-like sampling for better coverage
        candidates = []
        
        for _ in range(n_candidates):
            # Sample uniformly in trust region
            perturbation = np.random.uniform(-self.radius, self.radius, self.dim)
            candidate = self.center + perturbation
            # Clip to [0.01, 0.99] to avoid boundary issues
            candidate = np.clip(candidate, 0.01, 0.99)
            candidates.append(candidate)
        
        return np.array(candidates)
    
    def update(self, improved):
        """Update trust region based on whether we improved."""
        if improved:
            self.success_count += 1
            self.failure_count = 0
            if self.success_count >= 3:
                self.radius = min(self.radius * 1.5, self.max_radius)
                self.success_count = 0
        else:
            self.failure_count += 1
            self.success_count = 0
            if self.failure_count >= 3:
                self.radius = max(self.radius * 0.5, self.min_radius)
                self.failure_count = 0

## Main Optimization Loop

In [ ]:
def optimize_function(func_id, strategy='ensemble'):
    """
    Generate next query for a function using advanced techniques.
    
    Strategies:
    - 'ensemble': Multi-kernel GP ensemble with combined acquisitions
    - 'turbo': TuRBO-style trust region around best point
    - 'exploit': Pure exploitation with tiny trust region
    - 'explore': Broader exploration with larger radius
    """
    df = all_data[func_id]
    dim = len([c for c in df.columns if c.startswith('x')])
    
    X = df[[f'x{i}' for i in range(dim)]].values
    y = df['y'].values
    
    y_best = np.max(y)
    best_idx = np.argmax(y)
    x_best = X[best_idx]
    
    print(f"\n{'='*50}")
    print(f"Function {func_id} ({dim}D) - Strategy: {strategy}")
    print(f"Current best: {y_best:.6f} at {fmt(x_best)}")
    print(f"{'='*50}")
    
    # Build ensemble model
    ensemble = MultiKernelGPEnsemble(dim)
    ensemble.fit(X, y)
    
    # Determine trust region radius based on strategy
    if strategy == 'exploit':
        radius = 0.02  # Very tight
    elif strategy == 'turbo':
        radius = 0.05  # Moderate
    elif strategy == 'explore':
        radius = 0.15  # Broader
    else:  # ensemble
        radius = 0.08  # Balanced
    
    # Create trust region around best point
    tr = TuRBOTrustRegion(dim, x_best, initial_radius=radius)
    
    # Generate candidates
    n_candidates = 20000
    X_candidates = tr.generate_candidates(n_candidates)
    
    # Predict with ensemble
    mu, sigma = ensemble.predict(X_candidates)
    
    # Compute multiple acquisition functions
    ucb_values = ucb(mu, sigma, kappa=2.0)
    ei_values = expected_improvement(mu, sigma, y_best, xi=0.01)
    pi_values = probability_of_improvement(mu, sigma, y_best, xi=0.001)
    
    # Thompson sampling
    ts_idx = thompson_sampling(ensemble, X_candidates, n_samples=50)
    
    # Get best from each acquisition
    ucb_best_idx = np.argmax(ucb_values)
    ei_best_idx = np.argmax(ei_values)
    pi_best_idx = np.argmax(pi_values)
    
    # Voting: weight by predicted improvement
    candidates_best = [
        (X_candidates[ucb_best_idx], mu[ucb_best_idx], 'UCB'),
        (X_candidates[ei_best_idx], mu[ei_best_idx], 'EI'),
        (X_candidates[pi_best_idx], mu[pi_best_idx], 'PI'),
        (X_candidates[ts_idx], mu[ts_idx], 'TS'),
    ]
    
    print("\nAcquisition function recommendations:")
    for x, pred_mu, name in candidates_best:
        print(f"  {name}: {fmt(x)} -> pred: {pred_mu:.4f}")
    
    # Select based on strategy
    if strategy == 'exploit':
        # Use PI for pure exploitation
        x_next = X_candidates[pi_best_idx]
        method = 'PI (exploitation)'
    elif strategy == 'explore':
        # Use UCB for exploration
        x_next = X_candidates[ucb_best_idx]
        method = 'UCB (exploration)'
    else:
        # Ensemble: use EI as balanced choice
        x_next = X_candidates[ei_best_idx]
        method = 'EI (balanced)'
    
    # Final prediction at selected point
    mu_final, sigma_final = ensemble.predict(x_next.reshape(1, -1))
    
    print(f"\n>>> Selected ({method}): {fmt(x_next, 6)}")
    print(f"    Predicted: {mu_final[0]:.4f} ± {sigma_final[0]:.4f}")
    print(f"    Improvement potential: {mu_final[0] - y_best:.4f}")
    
    return x_next

## Function-Specific Strategies

Based on 7 weeks of data:
- **F1**: Best at 1.626, explore nearby (Week 7 at 0.854 was regression)
- **F2**: Stagnant at 0.667, try slightly different region
- **F3**: NEW BEST at -0.0145, exploit further!
- **F4**: Best at 0.600, need better exploitation
- **F5**: Best at 1618, very sensitive to x2, x3 near 1.0
- **F6**: NEW BEST at -0.681, continue exploiting
- **F7**: Close to 2.40, fine-tune
- **F8**: Near optimal at 9.915, very fine tuning

In [12]:
# Function-specific optimization
week8_queries = {}

# F1: Had breakthrough at [0.6346, 0.6356], Week 7 regressed
# Use tight trust region around best
week8_queries[1] = optimize_function(1, strategy='exploit')


Function 1 (2D) - Strategy: exploit
Current best: 1.626310 at [0.6346 0.6356]
  Kernel weights: ν=0.5: 0.206, ν=1.5: 0.366, ν=2.5: 0.429

Acquisition function recommendations:
  UCB: [0.6298 0.6292] -> pred: 1.5928
  EI: [0.6301 0.622 ] -> pred: 1.5950
  PI: [0.6307 0.6218] -> pred: 1.5970
  TS: [0.6342 0.6424] -> pred: 1.5522

>>> Selected (PI (exploitation)): [0.630734 0.621812]
    Predicted: 1.5970 ± 0.2176
    Improvement potential: -0.0293


In [13]:
# F2: Stagnant around [0.7026, 0.9266]
# Try slightly broader exploration
week8_queries[2] = optimize_function(2, strategy='turbo')


Function 2 (2D) - Strategy: turbo
Current best: 0.666983 at [0.7026 0.9266]
  Kernel weights: ν=0.5: 0.127, ν=1.5: 0.395, ν=2.5: 0.478


LinAlgError: SVD did not converge

In [ ]:
# F3: NEW BEST in Week 7! Exploit this promising region
week8_queries[3] = optimize_function(3, strategy='exploit')

In [ ]:
# F4: Best at [0.4046, 0.4148, 0.3574, 0.3990] = 0.600
# Balanced approach
week8_queries[4] = optimize_function(4, strategy='ensemble')

In [ ]:
# F5: Critical dimensions x2, x3 near 1.0
# Fine-tune with tight trust region
week8_queries[5] = optimize_function(5, strategy='exploit')

In [ ]:
# F6: NEW BEST! Continue exploiting
week8_queries[6] = optimize_function(6, strategy='exploit')

In [ ]:
# F7: Near 2.40, fine-tune the 6D space
week8_queries[7] = optimize_function(7, strategy='turbo')

In [ ]:
# F8: Very close to optimal, pure exploitation
week8_queries[8] = optimize_function(8, strategy='exploit')

## Summary and Submission

In [ ]:
print("\n" + "="*70)
print("WEEK 8 FINAL QUERIES")
print("="*70)

for func_id in range(1, 9):
    x = week8_queries[func_id]
    query_str = '-'.join([f"{v:.6f}" for v in x])
    current_best = best_values[func_id]
    print(f"\nFunction {func_id}:")
    print(f"  Query: {query_str}")
    print(f"  Current best: {current_best:.6f}")

In [ ]:
# Save submissions
for func_id in range(1, 9):
    x = week8_queries[func_id]
    query_str = '-'.join([f"{v:.6f}" for v in x])
    save_submission(func_id, query_str, module_name="Module 19 - Week 8")
    print(f"Saved F{func_id}: {query_str}")

## Part 2: Reflection on LLM-Centred Strategy

### 1. Prompt Patterns Used

**Primary Pattern: Structured Few-Shot with Chain-of-Thought**

I predominantly used a **structured few-shot prompting** approach where:
- Historical data (all 17 observations) served as implicit few-shot examples
- Each function's optimization history provided context for the model to learn patterns
- Chain-of-thought reasoning was embedded in the notebook structure itself

**Why this choice:**
- Zero-shot would ignore our valuable historical data
- Pure few-shot without structure led to inconsistent output formats
- Structured prompts with explicit sections (data loading → model fitting → acquisition → selection) ensured reproducible reasoning

**Simplified vs Structured Prompts:**
- *Simplified prompts* (e.g., "find the next best point") produced vague suggestions without mathematical grounding
- *Structured prompts* with explicit constraints (`[0.01, 0.99]` bounds, trust region radii, acquisition function formulas) produced actionable, precise queries
- The trade-off: structured prompts are verbose but eliminate ambiguity; simplified prompts are faster but require more post-processing

### 2. Temperature, Top-p, Top-k, and Max-Tokens Settings

**Settings Used (implicitly via Claude's defaults):**
- **Temperature**: Low (~0.3-0.5 effective) - prioritized coherent, deterministic optimization recommendations
- **Top-p**: ~0.9 - allowed some diversity in candidate generation while maintaining quality
- **Max-tokens**: Sufficient for full notebook execution (~4000 tokens per response)

**Trade-offs Observed:**
- *Lower temperature* → More exploitation-focused suggestions (repeating near-best points)
- *Higher temperature* → More exploration suggestions but occasionally nonsensical coordinates
- For BBO, I preferred **lower temperature** because:
  - We need precise floating-point coordinates
  - Exploration is already handled by acquisition functions (UCB, Thompson Sampling)
  - Hallucinated coordinates outside [0,1] would waste precious queries

**Effect on Query Selection:**
- Deterministic decoding ensured that running the same notebook twice produces identical queries
- This reproducibility is critical for scientific validity of the optimization process

### 3. Token Boundaries and Truncation Effects

**Observations:**
- **No truncation issues** with 17 data points - all observations fit comfortably in context
- Each function's data (~500-1500 tokens depending on dimensionality) is well within limits

**How I Checked:**
1. Monitored total token count: 8 functions × ~200 tokens/function = ~1600 tokens for data alone
2. Full notebook context including code: ~8000 tokens total
3. Claude's context window (200K) handles this easily

**Token Boundary Concerns:**
- Floating-point numbers like `0.634586` tokenize as multiple tokens (`0`, `.`, `634`, `586`)
- This *could* cause issues with very precise coordinates, but 6 decimal places proved sufficient
- I avoided scientific notation (e.g., `1.23e-10`) which tokenizes poorly and can confuse the model

**Potential Issues for Future:**
- At 50+ data points per function, context could become a concern
- Would need to implement sliding windows or summarization strategies

### 4. Limitations with 17 Data Points

**Prompt Overfitting:**
- With only 17 points in 2-8 dimensional spaces, GP models can overfit
- Mitigated by using **multi-kernel ensembles** that average across different smoothness assumptions
- Observed: F4's kernel strongly preferred ν=1.5 (92% weight), suggesting potential overfitting to that smoothness

**Attention on Irrelevant Context:**
- Early poor observations (e.g., F1's initial random samples with values ~10^-80) could mislead attention
- Solution: StandardScaler normalization ensures all observations contribute proportionally
- The model correctly focused on the high-value region around [0.63, 0.64] for F1

**Diminishing Returns:**
- F1, F5, F8: Near optimal values - additional queries show diminishing improvements
- F2: Stagnant despite varied queries - may have found local optimum or need different exploration strategy
- With only 17 points, we haven't exhausted the information gain potential for most functions

### 5. Hallucination Reduction Strategies

**Strategies Employed:**

1. **Constrained Output Format:**
   - Queries must be hyphen-separated decimals: `0.630734-0.621812`
   - Explicit bounds checking: `np.clip(candidate, 0.01, 0.99)`
   - Format validation before submission

2. **Retrieval of Prior Information:**
   - Always loaded historical `samples.csv` before generating new queries
   - Cross-referenced best known values to ensure suggestions are improvements
   - Used GP predictions as "ground truth" rather than LLM intuition

3. **Tighter Instructions:**
   - Explicit acquisition function formulas (UCB, EI, PI) leave no room for interpretation
   - Trust region radii specified numerically (0.02 for exploit, 0.05 for turbo)
   - No natural language "suggest a good point" - only mathematical optimization

4. **Ensemble Verification:**
   - Multiple acquisition functions vote on the best candidate
   - If UCB, EI, and PI all agree, higher confidence in the suggestion
   - Thompson Sampling provides stochastic verification

**Result:** Zero hallucinated coordinates across 7 weeks of submissions.

### 6. Scaling Strategies for Future Rounds

**For Larger Datasets (50+ points):**
- **Summarization**: Compress historical data to key statistics (best points, gradient estimates, Pareto frontier)
- **Sliding Window**: Only include most recent N observations in prompt
- **Hierarchical Prompting**: First identify promising regions, then zoom in with detailed prompts

**For More Complex LLMs:**
- **Multi-agent Approaches**: Separate agents for exploration vs exploitation
- **Tool Use**: Let LLM call GP fitting and acquisition optimization as external tools
- **Iterative Refinement**: Generate candidate → Evaluate → Refine cycle within single query budget

**Computational Constraints:**
- GP fitting scales O(n³) - would need sparse approximations for 100+ points
- Consider neural network surrogates (already implemented) for faster inference
- Batch candidate evaluation to amortize model overhead

### 7. Practitioner Mindset: Balancing Exploration, Risk, and Constraints

**Key Design Choices and Their Rationale:**

| Choice | Exploration | Risk | Constraint |
|--------|-------------|------|------------|
| Multi-kernel ensemble | Handles unknown smoothness | Reduces model misspecification risk | Moderate compute cost |
| Trust regions | Limits to promising areas | Prevents catastrophic exploration | Controls search space |
| Multiple acquisitions | UCB explores, PI exploits | Voting reduces single-method risk | Minimal overhead |
| Thompson Sampling | Natural exploration via randomness | Bounded by posterior | Sampling is cheap |

**Black-Box Mindset:**
- We cannot see the function, so **every assumption could be wrong**
- Multi-kernel ensembles hedge against smoothness assumptions
- Trust regions hedge against global vs local optima
- The 1 query/week constraint forces us to be **maximally informative** with each observation

**Risk Management:**
- Never submitted untested query formats (validated all submissions)
- Maintained best-known-point tracking to avoid regression
- Used conservative trust regions (0.02-0.05) to avoid wild exploration late in optimization

**What I Learned:**
- BBO is fundamentally about **information acquisition under uncertainty**
- LLM prompting and BBO share the same core challenge: making good decisions with incomplete information
- Structured, reproducible approaches outperform ad-hoc intuition

---

## High-Level Summary: LLM Foundations and BBO Strategy

After 8 weeks and 17 data points, several key insights emerge about integrating LLM-based reasoning with black-box optimization:

### The Core Tension: Structure vs. Flexibility

The fundamental challenge was balancing **structured mathematical optimization** (GPs, acquisition functions, trust regions) with **flexible LLM reasoning** (pattern recognition, context understanding, adaptive strategy selection). 

**Resolution:** Use LLMs for high-level strategy selection and code generation, but delegate numerical optimization to deterministic algorithms. This hybrid approach gets the best of both worlds—human-like reasoning about which approach to use, with mathematical rigor in execution.

### What Worked

1. **Multi-kernel GP ensembles** automatically adapted to each function's smoothness without manual tuning
2. **Trust regions** prevented catastrophic exploration while allowing refinement
3. **Structured prompts** with explicit mathematical formulas eliminated hallucination risk
4. **Reproducible notebooks** ensured scientific validity and debugging capability

### What Surprised Me

- **F3 and F6 breakthroughs in Week 7** came from persistent exploration of seemingly stagnant regions—patience pays off in BBO
- **Kernel weight distributions** revealed hidden structure (F5 is rough, F8 is smooth) that pure intuition missed
- **17 points is enough** for reliable GP fits in low dimensions, but ensemble methods remain valuable for uncertainty

### The LLM-BBO Parallel

Both LLM prompting and BBO require making decisions under uncertainty:
- In prompting: we don't know how the model will interpret our instructions
- In BBO: we don't know the true function landscape

The same principles apply to both:
- **Hedge your bets** (multi-kernel ensembles ≈ multiple prompt variations)
- **Exploit what works** (trust regions ≈ few-shot examples from successful patterns)
- **Explore carefully** (Thompson Sampling ≈ temperature-based diversity)

### Looking Forward

For the remaining weeks, the strategy is clear:
- **Exploit** functions showing improvement (F3, F6, F7, F8)
- **Accept** potentially optimal values for stable functions (F1, F5)
- **Consider reset** for stagnant functions (F2) if no improvement by Week 9

The interplay between LLM capabilities and classical optimization theory has been the most valuable learning outcome—understanding when to trust algorithmic recommendations versus when human intuition (or LLM reasoning) should override them.